# **CuffLessAI - Non-invasive Blood Pressure Detection and Prediction**

## Part 1 - Data loading & Inspection

In [ ]:
import pandas as pd 
import mat73 as mat
import numpy as np
import scipy.signal as signal

In [ ]:
p1 = mat.loadmat("data/Part_1.mat") 
p2 = mat.loadmat("data/Part_2.mat") 
p3 = mat.loadmat("data/Part_3.mat") 
p4 = mat.loadmat("data/Part_4.mat") 

In [ ]:
p1
##test_df = pd.DataFrame()

##### **Initial Reflection**

Cell array matricies appear as a dictionary of nested arrays. The consistent rows indicate one of each of the rows corresponds to ppg, abp, and ecg signals. The "columns" however in these shape metrics are possibly measurements that are not yet separated into windows or times

#### **Preprocessing guidelines (See github description)**

When you load the MATLAB files, you might want to consider processing them into 5-second windows. When you process the data into 5-second window, each window should contain 625 PPG samples as input and a pair of blood pressure values as labels.

For each window, you can extract systolic and diastolic pressure from the ABP signal by finding the peaks (systolic) and valleys (diastolic). If a window contains noisy signals or physiologically impossible values, you can consider discarding the noisy data.

After pre-processing all the data files, you will have a dataset with 10,000+ paired dataset. Each example is a 625-sample PPG window, and the systolic and diastolic labels.


625 = 125 (sampling freq) * 5 (second window) so every 125 freqs is a 5 second window


In [ ]:
def preprocess(file : dict) -> pd.DataFrame:
    '''
    Data type input is a dict with a cell array matrix structure. Multiple arrays within which arrays are further nested 
    in a shape of (3,x) 3 for ppg, abp, ecg and x representing data poitns
    Data points are classified into windows the problems statement states 625 since 125*5 = 625
    Divy x data points by the sampling frequency * time frame 
    Use SciPy Signal to find peaks 
    Append to window array (initialized in the beginning) and cast to a dataframe


    Later this will be concatenated
    '''
    window = []
    for part, arrs in file.items():
        for array_idx, arr in enumerate(arrs):
            ##breaking the cell of arrays setup into their array sections (based on observation all follow shape (3,x))
            ppg, abp, ecg = arr[0], arr[1], arr[2]
            n_points = arr.shape[1]

            ##determinign 5 second windows
            n_windows = n_points // 625

            ##splitting into windows of datapoints and add per window to a dataframe
            for data in range(n_windows):
                begin = data * 625 
                end = begin + 625

                ppg_signals = ppg[begin:end]
                abp_signals = abp[begin:end]
                ecg_signals = ecg[begin:end]


                ##use scipy to find location indices of peaks
                systolic_indices, _ = signal.find_peaks(abp_signals)
                diastolic_indices, _ = signal.find_peaks(-abp_signals) ##negative to find minima

                systolic  = np.median(abp_signals[systolic_indices])
                diastolic = np.median(abp_signals[diastolic_indices])

                window.append({
                    'source': file,
                    'array_idx': array_idx,
                    'window': data,
                    'ppg': ppg_signals,
                    'abp': abp_signals,
                    'ecg': ecg_signals,
                    'systolic': systolic,
                    'diastolic': diastolic,
                })

    return pd.DataFrame(window)
                

In [ ]:
(preprocess(p1)).shape

In [ ]:
(preprocess(p2)).shape